In [39]:
import pandas as pd
from openai import AzureOpenAI
import os
from dotenv import load_dotenv
import ast

In [9]:
column_converters = {"ingredients" : ast.literal_eval,
                     "directions" : ast.literal_eval,
                     "NER" : ast.literal_eval}

In [10]:
df = pd.read_csv("../data/dataset_processed.csv", converters = column_converters)

In [11]:
df.head(5)

,title,ingredients,directions,link,source,NER
0,Marinated Flank Steak Recipe,"[1 1/2 pound flank steak, 1/2 c. finely minced...","[Remove tenderloin from steak., Score meat., C...",cookeatshare.com/recipes/marinated-flank-steak...,Recipes1M,"[flank steak, green onions, red wine, soy sauc..."
1,French Chicken Stew,"[1 tablespoon rosemary, 1 teaspoon thyme, 3 ba...",[combine all ingredients in slow cooker (6 qua...,www.yummly.com/recipe/French-Chicken-Stew-1433580,Gathered,"[rosemary, thyme, bay leaves, paprika, pepper,..."
2,Glazed Carrots,"[3 to 4 carrots, 1 1/2 Tbsp. butter, 1/3 c. br...",[Cook 3 to 4 carrots; cut crosswise in 1-inch ...,www.cookbooks.com/Recipe-Details.aspx?id=1011892,Gathered,"[carrots, butter, brown sugar, lemon rind]"
3,Moms Pie Dough,"[4.5 Cups Flour, 1.5 Tsp Salt, Pinch Baking Po...","[Mix all dry ingredients in a bowl., , Add cri...",www.epicurious.com/recipes/member/views/moms-p...,Gathered,"[Flour, Salt, Baking Powder, Sugar, Crisco, eg..."
4,Pretzel Salad Or Dessert,"[2 c. crushed small thin pretzels (sticks), 3/...","[Mix and press in baking pan, approximately 13...",www.cookbooks.com/Recipe-Details.aspx?id=106723,Gathered,"[thin pretzels, margarine]"


In [12]:
texts = df.apply(lambda row: f"{row['title']} | {', '.join(row['NER'])}", axis=1).tolist()

In [13]:
texts[:5]

['Marinated Flank Steak Recipe | flank steak, green onions, red wine, soy sauce, salad oil, sesame seeds, brown sugar, grnd black pepper, grnd ginger, clove garlic',
 'French Chicken Stew | rosemary, thyme, bay leaves, paprika, pepper, red wine, chicken broth, button mushrooms, mushroom mix, carrots, onion, frozen green beans, black olives, handful grape tomatoes, chicken, stalks celery, water',
 'Glazed Carrots | carrots, butter, brown sugar, lemon rind',
 'Moms Pie Dough  | Flour, Salt, Baking Powder, Sugar, Crisco, egg, vinegar, Water',
 'Pretzel Salad Or Dessert | thin pretzels, margarine']

In [44]:
endpoint = "https://rag-recipe-resource.cognitiveservices.azure.com/"
deployment = "text-embedding-3-large"
api_version = "2024-02-01"
load_dotenv()
API_KEY = os.getenv('API_KEY_AZURE')

In [45]:
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=API_KEY,
    api_version=api_version
)

In [19]:
batch_size = 100

In [20]:
embeddings = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    response = client.embeddings.create(
        input=batch,
        model=deployment
    )
    embeddings.extend([item.embedding for item in response.data])

In [22]:
len(embeddings)

10000

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        10000 non-null  object
 1   ingredients  10000 non-null  object
 2   directions   10000 non-null  object
 3   link         10000 non-null  object
 4   source       10000 non-null  object
 5   NER          10000 non-null  object
 6   embedding    10000 non-null  object
dtypes: object(7)
memory usage: 547.0+ KB


In [32]:
new_df = df.drop(['link', 'source'], axis=1)

In [33]:
new_df['embedding'] = embeddings

In [47]:
new_df.to_json("../data/dataset_embeddings.json", orient='records', indent=4)